In [1]:
import pandas as pd

C:\Users\JorgeAZ\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [5]:
df = pd.read_csv("rvvcca.-datos-horarios-valencia-2016-2021-curt-cas.csv", sep=";")
df.head()

,Fecha,Día de la semana,Día del mes,Hora,Estación,PM1,PM2.5,PM10,NO,NO2,...,C7H8,C6H6,Ruido,C8H10,Temperatura,Humedad relativa,Presión,Radiación,Precipitación,Velocidad máxima del viento
0,2020-01-01T00:00:00,Miércoles,1,0:00:00,Avda. Francia,NaN,77.0,83.0,16.0,45.0,...,NaN,NaN,NaN,NaN,6.6,89.0,1019.0,0.0,0.0,3.2
1,2020-01-01T00:00:00,Miércoles,1,1:00:00,Avda. Francia,NaN,65.0,70.0,20.0,37.0,...,NaN,NaN,NaN,NaN,6.2,89.0,1019.0,0.0,0.0,4.2
2,2020-01-01T00:00:00,Miércoles,1,2:00:00,Avda. Francia,NaN,47.0,51.0,16.0,35.0,...,NaN,NaN,NaN,NaN,5.9,90.0,1019.0,0.0,0.0,4.1
3,2020-01-01T00:00:00,Miércoles,1,3:00:00,Avda. Francia,NaN,43.0,46.0,16.0,29.0,...,NaN,NaN,NaN,NaN,5.5,90.0,1019.0,0.0,0.0,4.1
4,2020-01-01T00:00:00,Miércoles,1,4:00:00,Avda. Francia,NaN,36.0,39.0,10.0,23.0,...,NaN,NaN,NaN,NaN,5.3,89.0,1018.0,0.0,0.0,3.8


In [7]:
df.shape

(449026, 27)

In [9]:
df.columns

Index(['Fecha', 'Día de la semana', 'Día del mes', 'Hora', 'Estación', 'PM1',
       'PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'O3', 'SO2', 'CO',
       'Velocidad del viento', 'Dirección del viento', 'NH3', 'C7H8', 'C6H6',
       'Ruido', 'C8H10', 'Temperatura', 'Humedad relativa', 'Presión',
       'Radiación', 'Precipitación', 'Velocidad máxima del viento'],
      dtype='str')

In [11]:
df["Estación"].value_counts()

Estación
Avda. Francia                52608
Bulevard Sud                 52608
Pista Silla                  52608
Politécnico                  52608
Viveros                      52608
Consellería Meteo            52608
Molí del Sol                 43848
Valencia Centro              28162
Puerto Valencia              26304
Nazaret Meteo                17544
Puerto Moll Trans. Ponent     8760
Puerto llit antic Túria       8760
Name: count, dtype: int64

In [13]:
df_est = df[df["Estación"] == "Avda. Francia"]
df_est.shape

(52608, 27)

In [15]:
df_est["NO2"].isna().sum()

2606

In [17]:
columnas = [
    "Fecha", "Día de la semana", "Hora",
    "NO2",
    "Velocidad del viento", "Temperatura", "Humedad relativa", "Precipitación"
]
df_limpio = df_est[columnas]
df_limpio.head()

,Fecha,Día de la semana,Hora,NO2,Velocidad del viento,Temperatura,Humedad relativa,Precipitación
0,2020-01-01T00:00:00,Miércoles,0:00:00,45.0,1.3,6.6,89.0,0.0
1,2020-01-01T00:00:00,Miércoles,1:00:00,37.0,1.9,6.2,89.0,0.0
2,2020-01-01T00:00:00,Miércoles,2:00:00,35.0,2.2,5.9,90.0,0.0
3,2020-01-01T00:00:00,Miércoles,3:00:00,29.0,1.7,5.5,90.0,0.0
4,2020-01-01T00:00:00,Miércoles,4:00:00,23.0,1.2,5.3,89.0,0.0


In [19]:
df_limpio.isna().sum()

Fecha                      0
Día de la semana           0
Hora                       0
NO2                     2606
Velocidad del viento    1334
Temperatura             1154
Humedad relativa        2975
Precipitación            920
dtype: int64

In [21]:
df_final = df_limpio.dropna()
df_final.shape

(46246, 8)

In [23]:
df_final = df_final.copy()
df_final["FechaHora"] = pd.to_datetime(df_final["Fecha"].str[:10] + " " + df_final["Hora"])
df_final = df_final.sort_values("FechaHora").reset_index(drop=True)
df_final.head()

,Fecha,Día de la semana,Hora,NO2,Velocidad del viento,Temperatura,Humedad relativa,Precipitación,FechaHora
0,2016-01-01T00:00:00,Viernes,0:00:00,28.0,3.4,13.3,79.0,0.0,2016-01-01 00:00:00
1,2016-01-01T00:00:00,Viernes,2:00:00,48.0,0.5,12.9,79.0,0.0,2016-01-01 02:00:00
2,2016-01-01T00:00:00,Viernes,3:00:00,58.0,0.2,12.6,81.0,0.0,2016-01-01 03:00:00
3,2016-01-01T00:00:00,Viernes,4:00:00,57.0,0.6,12.5,83.0,0.0,2016-01-01 04:00:00
4,2016-01-01T00:00:00,Viernes,5:00:00,53.0,0.6,12.4,83.0,0.0,2016-01-01 05:00:00


In [25]:
# 1) Variable objetivo: ¿el NO2 supera el umbral de alerta (40)?  1 = sí, 0 = no
df_final["alerta"] = (df_final["NO2"] > 40).astype(int)

# 2) Pista estrella: el NO2 de la hora anterior
df_final["NO2_anterior"] = df_final["NO2"].shift(1)

df_final[["FechaHora", "NO2", "NO2_anterior", "alerta"]].head()

,FechaHora,NO2,NO2_anterior,alerta
0,2016-01-01 00:00:00,28.0,NaN,0
1,2016-01-01 02:00:00,48.0,28.0,1
2,2016-01-01 03:00:00,58.0,48.0,1
3,2016-01-01 04:00:00,57.0,58.0,1
4,2016-01-01 05:00:00,53.0,57.0,1


In [27]:
# Quitar la primera fila, que tiene NO2_anterior vacío
df_modelo = df_final.dropna(subset=["NO2_anterior"]).reset_index(drop=True)

# Definir las pistas (X) y la respuesta (y)
features = ["NO2_anterior", "Velocidad del viento", "Temperatura", "Humedad relativa", "Precipitación"]
X = df_modelo[features]
y = df_modelo["alerta"]

print(X.shape, y.shape)
y.value_counts()

(46245, 5) (46245,)


alerta
0    38050
1     8195
Name: count, dtype: int64

In [29]:
# Punto de corte: 80% más antiguo para entrenar, 20% más reciente para test
corte = int(len(df_modelo) * 0.8)

X_train = X.iloc[:corte]
X_test  = X.iloc[corte:]
y_train = y.iloc[:corte]
y_test  = y.iloc[corte:]

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Fechas train:", df_modelo['FechaHora'].iloc[0], "→", df_modelo['FechaHora'].iloc[corte-1])
print("Fechas test :", df_modelo['FechaHora'].iloc[corte], "→", df_modelo['FechaHora'].iloc[-1])

Train: (36996, 5)  Test: (9249, 5)
Fechas train: 2016-01-01 02:00:00 → 2020-09-29 09:00:00
Fechas test : 2020-09-29 10:00:00 → 2021-12-31 23:00:00


In [31]:
from sklearn.metrics import classification_report, roc_auc_score

# Baseline: predigo alerta (1) si el NO2 de la hora anterior ya superaba 40
pred_baseline = (X_test["NO2_anterior"] > 40).astype(int)

print(classification_report(y_test, pred_baseline, digits=3))
print("AUC:", roc_auc_score(y_test, pred_baseline))

              precision    recall  f1-score   support

           0      0.981     0.981     0.981      8793
           1      0.635     0.636     0.635       456

    accuracy                          0.964      9249
   macro avg      0.808     0.808     0.808      9249
weighted avg      0.964     0.964     0.964      9249

AUC: 0.8084862659890941


In [33]:
from sklearn.ensemble import RandomForestClassifier

modelo = RandomForestClassifier(
    n_estimators=200,        # nº de árboles en el bosque
    class_weight="balanced", # compensa el desbalanceo (da más peso a las alertas, la clase rara)
    random_state=42,         # para que el resultado sea reproducible
    n_jobs=-1                # usa todos los núcleos del procesador (más rápido)
)
modelo.fit(X_train, y_train)

pred = modelo.predict(X_test)
print(classification_report(y_test, pred, digits=3))
print("AUC:", roc_auc_score(y_test, modelo.predict_proba(X_test)[:, 1]))

              precision    recall  f1-score   support

           0      0.979     0.978     0.978      8793
           1      0.581     0.592     0.586       456

    accuracy                          0.959      9249
   macro avg      0.780     0.785     0.782      9249
weighted avg      0.959     0.959     0.959      9249

AUC: 0.9490390332421523


In [35]:
from sklearn.metrics import f1_score
import numpy as np

probs = modelo.predict_proba(X_test)[:, 1]   # probabilidades de alerta

# Probar muchos umbrales y quedarnos con el de mejor F1 en la clase 1
umbrales = np.arange(0.1, 0.9, 0.05)
f1s = [f1_score(y_test, (probs >= u).astype(int)) for u in umbrales]
mejor = umbrales[np.argmax(f1s)]

print("Mejor umbral:", round(mejor, 2), " F1:", round(max(f1s), 3))
print(classification_report(y_test, (probs >= mejor).astype(int), digits=3))

Mejor umbral: 0.55  F1: 0.588
              precision    recall  f1-score   support

           0      0.977     0.983     0.980      8793
           1      0.625     0.555     0.588       456

    accuracy                          0.962      9249
   macro avg      0.801     0.769     0.784      9249
weighted avg      0.960     0.962     0.961      9249



In [37]:
# 1) Extraer la hora como número entero (0–23) desde el texto "H:00:00"
df_modelo["hora_num"] = df_modelo["Hora"].str.split(":").str[0].astype(int)

# 2) Reconstruir X con la nueva pista incluida
features = ["NO2_anterior", "hora_num",
            "Velocidad del viento", "Temperatura", "Humedad relativa", "Precipitación"]
X = df_modelo[features]
y = df_modelo["alerta"]

# 3) Rehacer el corte temporal (igual que antes: 80% pasado / 20% futuro)
corte = int(len(df_modelo) * 0.8)
X_train, X_test = X.iloc[:corte], X.iloc[corte:]
y_train, y_test = y.iloc[:corte], y.iloc[corte:]

# 4) Reentrenar el mismo Random Forest
modelo = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                random_state=42, n_jobs=-1)
modelo.fit(X_train, y_train)

# 5) Evaluar
pred = modelo.predict(X_test)
print(classification_report(y_test, pred, digits=3))
print("AUC:", roc_auc_score(y_test, modelo.predict_proba(X_test)[:, 1]))

              precision    recall  f1-score   support

           0      0.982     0.981     0.982      8793
           1      0.639     0.660     0.649       456

    accuracy                          0.965      9249
   macro avg      0.811     0.820     0.815      9249
weighted avg      0.965     0.965     0.965      9249

AUC: 0.9680104638657943


In [39]:
import joblib

# Guardamos el modelo y la lista de features juntos, en un solo archivo
joblib.dump({"modelo": modelo, "features": features}, "modelo_no2.joblib")
print("Guardado:", features)

Guardado: ['NO2_anterior', 'hora_num', 'Velocidad del viento', 'Temperatura', 'Humedad relativa', 'Precipitación']


In [42]:
# Guardamos una versión reducida para la app (solo lo necesario para el drift)
df_modelo[["FechaHora", "NO2"]].to_csv("datos_drift.csv", index=False)
print("Guardado datos_drift.csv")

Guardado datos_drift.csv


In [45]:
from sklearn.metrics import f1_score, recall_score, precision_score, roc_auc_score

pred = modelo.predict(X_test)
proba = modelo.predict_proba(X_test)[:, 1]

metricas = {
    "AUC": round(roc_auc_score(y_test, proba), 3),
    "F1 (alerta)": round(f1_score(y_test, pred), 3),
    "Recall (alerta)": round(recall_score(y_test, pred), 3),
    "Precision (alerta)": round(precision_score(y_test, pred), 3),
}

# Volvemos a guardar el modelo, ahora incluyendo las métricas
joblib.dump({"modelo": modelo, "features": features, "metricas": metricas},
            "modelo_no2.joblib")
print(metricas)

{'AUC': 0.968, 'F1 (alerta)': 0.649, 'Recall (alerta)': 0.66, 'Precision (alerta)': 0.639}
